# 01 - Train Baseline MobileNetV2 on CIFAR-10

This notebook trains a baseline MobileNetV2 model on the CIFAR-10 dataset.  
It evaluates the model's accuracy and saves the trained model for further optimization.


In [1]:

# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# 2. Import Libraries
import torch
import torchvision
import torchvision.transforms as transforms
import torchvision.models as models
import torch.nn as nn
import torch.optim as optim
import os
from time import time


Mounted at /content/drive


In [2]:
# 3 Setup drvice
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")




Using device: cpu


In [3]:
# 4. Load and Transform CIFAR-10
import torchvision.datasets as datasets
import torch.utils.data as data

# Data Augmentation for Training
train_transform = transforms.Compose([
    transforms.Resize(224),                      # Resize for MobileNet
    transforms.RandomHorizontalFlip(p=0.5),      # Horizontal flip (mirroring)
    transforms.RandomRotation(15),               # Rotate ±15 degrees
    transforms.ColorJitter(0.2, 0.2, 0.2),        # Brightness, contrast, saturation
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

# No augmentation for testing
test_transform = transforms.Compose([
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

# Download dataset directly into Colab
trainset = datasets.CIFAR10(root='./data', train=True, download=True, transform=train_transform)
trainloader = data.DataLoader(trainset, batch_size=32, shuffle=True)

testset = datasets.CIFAR10(root='./data', train=False, download=True, transform=test_transform)
testloader = data.DataLoader(testset, batch_size=32, shuffle=False)

print(" CIFAR-10 loaded successfully!")


100%|██████████| 170M/170M [00:01<00:00, 103MB/s]


 CIFAR-10 loaded successfully!


In [4]:
# 5. Load and Modify MobileNetV2
model = models.mobilenet_v2(pretrained=True)
model.classifier[1] = nn.Linear(model.last_channel, 10)  # CIFAR-10 = 10 classes
model = model.to(device)



/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=MobileNet_V2_Weights.IMAGENET1K_V1`. You can also use `weights=MobileNet_V2_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/mobilenet_v2-b0353104.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v2-b0353104.pth
100%|██████████| 13.6M/13.6M [00:00<00:00, 87.9MB/s]


In [5]:
# 6. Loss Function and Optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)



In [6]:
# 7. Training Loop
EPOCHS = 2
max_batches = 20  # Limit training to 20 batches

for epoch in range(EPOCHS):
    running_loss = 0.0
    model.train()

    for i, (images, labels) in enumerate(trainloader):
        if i >= max_batches:
            break

        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f" Epoch {epoch+1}/{EPOCHS}, Loss: {running_loss:.4f}")




 Epoch 1/2, Loss: 32.2652
 Epoch 2/2, Loss: 26.8707


In [7]:
# 8. Evaluate Accuracy
correct = 0
total = 0
model.eval()

with torch.no_grad():
    for images, labels in testloader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total
print(f" Test Accuracy: {accuracy:.2f}%")



 Test Accuracy: 58.26%


In [8]:
# 9. Save the Trained Model to Google Drive
save_path = '/content/drive/MyDrive/AI_MODEL_OPTIMIZATION/models'
os.makedirs(save_path, exist_ok=True)
torch.save(model.state_dict(), os.path.join(save_path, 'mobilenetv2_cifar10_baseline.pth'))
print(f" Model saved to: {save_path}/mobilenetv2_cifar10_baseline.pth")


 Model saved to: /content/drive/MyDrive/AI_MODEL_OPTIMIZATION/models/mobilenetv2_cifar10_baseline.pth
